In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import glob
import polars as pl
import json

# elapid (MaxEnt / IWLR) + features + spatial CV
import elapid
from elapid import features as fe
from elapid import GeographicKFold

from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    precision_score, recall_score, log_loss, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance

# Winter: Dec (12), Jan (1), Feb (2) -> 0
# Spring: Mar..May (3..5) -> 1
# Summer: Jun..Aug (6..8) -> 2
# Fall:   Sep..Nov (9..11) -> 3

In [ ]:
parambasedir =  '/mnt/f/readyparams/param_csvs'
baseoutputdir = '/mnt/f/readyparams'
sp = 'Protonotaria_citrea'

In [ ]:
# Paths
sp = sp.lower()

if "ppp_paramsoutput" in baseoutputdir:
    outputdir = baseoutputdir
else:
    outputdir = os.path.join(baseoutputdir,'ppp_paramsoutput')
    os.makedirs(outputdir, exist_ok=True)
    outputdir = os.path.join(outputdir,sp)
    os.makedirs(outputdir, exist_ok=True)
    
print(outputdir)

pixel_area = 0.01

In [ ]:
# === Load Presence Points ===

# Find matching files
csv_files = glob.glob(os.path.join(parambasedir, f"*{sp}*.csv"))

dfs = []
for file_path in csv_files:
    print(f"Reading: {file_path}")

    # Read CSV
    df = pl.read_csv(file_path)
    # Extract lon/lat
    df = df.with_columns([
        pl.col(".geo").map_elements(lambda x: json.loads(x)["coordinates"][0], return_dtype=pl.Float64).alias("longitude"),
        pl.col(".geo").map_elements(lambda x: json.loads(x)["coordinates"][1], return_dtype=pl.Float64).alias("latitude")
    ])
    
    # Drop unnecessary columns
    df = df.drop(["system:index", ".geo"])
    
    
    dfs.append(df)
    
combined_df = pl.concat(dfs, how='diagonal')

combined_df = combined_df.with_columns(
    pl.lit(1).alias("label")  # constant string
)
len(combined_df)

In [ ]:
# Ensure 'month' column exists
if "month" not in combined_df.columns:
    combined_df = combined_df.with_columns(
        pl.col("obs_date")
        .str.strptime(pl.Date, format="%Y-%m-%d")  # ✅ use 'format' instead of 'fmt'
        .dt.month()
        .alias("month")
    )

# Count rows by month
month_counts = (
    combined_df
    .group_by("month")
    .len()
    .sort("month")
)

# Convert to pandas for plotting
month_counts_pd = month_counts.to_pandas()

# Plot
plt.figure(figsize=(8, 5))
plt.bar(month_counts_pd["month"], month_counts_pd["len"], color="skyblue")
plt.title(f"Row Count by Month for {sp}")
plt.xlabel("Month")
plt.ylabel("Row Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(outputdir,f"CountByMonth_{sp}.png"))
plt.show()

In [ ]:
csv_files = glob.glob(os.path.join(parambasedir, f"*background*.csv"))

bgs = []
for file_path in csv_files:
    print(f"Reading: {file_path}")

    # Read CSV
    bg = pl.read_csv(file_path)
    # Extract lon/lat
    bg = bg.with_columns([
        pl.col(".geo").map_elements(lambda x: json.loads(x)["coordinates"][0], return_dtype=pl.Float64).alias("longitude"),
        pl.col(".geo").map_elements(lambda x: json.loads(x)["coordinates"][1], return_dtype=pl.Float64).alias("latitude")
    ])
    
    # Drop unnecessary columns
    bg = bg.drop(["system:index", ".geo"])
    
    
    bgs.append(bg)
    
combined_bg = pl.concat(bgs, how='diagonal')
combined_bg = combined_bg.with_columns(
    pl.lit(0).alias("label")  # constant string
)
len(combined_bg)

In [ ]:
# Count rows by month
combined_bg = combined_bg.with_columns(
    pl.col("obs_date").str.to_datetime(strict=False).dt.month().alias("month")
)

month_counts = (
    combined_bg
    .group_by("month")
    .len()
    .sort("month")  # optional: sort chronologically if month is numeric or formatted
)

# Convert to pandas for plotting
month_counts_pd = month_counts.to_pandas()

# Plot
plt.figure(figsize=(8, 5))
plt.bar(month_counts_pd["month"], month_counts_pd["len"], color="skyblue")
plt.title(f"Background acount by month")
plt.xlabel("Month")
plt.ylabel("Row Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------
# === Prepare Data ===
# ------------------------
print(combined_df.columns)
print(combined_bg.columns)

data = pd.concat([combined_df.to_pandas(), combined_bg.to_pandas()], ignore_index=True)
print(data.columns)
print(len(data))

# Keep coords for spatial CV
coords = data[["longitude", "latitude"]].to_numpy()

# Drop non-predictor columns (keep coords only in 'coords', not in X)
X = data.drop(columns=[
    'basisofrecord', 'coordinateuncertaintyinmeters', 'species', 'rand',
    'date', 'day', 'obs_date', 'observation_date', 'protocol_name',
    'scientific_name', 'year', 'longitude', 'latitude', 'label', 'month',
    'srad_mean_mean', 'swe_mean_mean', 'vp_mean_mean'
], errors='ignore').copy()

# Labels: 1 = presence, 0 = background (PPP/MaxEnt presence-background)
y = data['label'].astype(int)
X = X.fillna(0)
X.columns = [col.replace('_mean_mean', '_10000m') for col in X.columns]
print(X.columns)

In [ ]:
def fit_maxent_with_tuning(
    X: "pd.DataFrame",
    y: "pd.Series",
    coords: "np.ndarray",
    outputdir: str,
    sp: str,
    selection_metric: str = "best_f1",  # or "mean_auc"
    base_tau: float = 0.5,
    n_splits: int = 5,
    random_state: int = 42,
    reg_values: "list[float]" = None,
    feature_types: "list[str]" = None,
    n_cpus: int = 4,
):

    # ---------- setup ----------
    os.makedirs(outputdir, exist_ok=True)

    # Work on copies; avoid chained assignment warnings
    X = X.copy()
    y = y.astype(int).copy()

    n_pres = int(y.sum())

    # Conservative features for very small n (mirror MaxEnt guidance)
    if feature_types is None:
        feature_types = ["linear", "quadratic"] if n_pres < 10 else ["linear", "quadratic", "hinge"]

    # Wider sweep for small n
    if reg_values is None:
        reg_values = [1.5, 2.0, 3.0, 4.0] if n_pres < 10 else [0.75, 1.0, 1.5, 2.0, 3.0]

    # ---------- build spatial folds ONCE (preferred); fallback to stratified ----------
    try:
        from elapid import GeographicKFold
        gkf = GeographicKFold(n_splits=n_splits, random_state=random_state)
        folds = list(gkf.split(X, y, coords=coords))
        cv_name = "GeographicKFold"
    except Exception:
        folds = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state).split(X, y))
        cv_name = "StratifiedKFold"

    # ---------- regularization sweep ----------
    reg_results = []

    for reg in reg_values:
        model_alpha = elapid.MaxentModel(
            feature_types=feature_types,
            tau=base_tau,
            clamp=True,
            scorer="roc_auc",           # internal elapid scorer for its own CV
            beta_multiplier=reg,
            beta_lqp=1.0,
            beta_hinge=1.0,
            beta_threshold=1.0,
            beta_categorical=1.0,
            n_hinge_features=10,
            n_threshold_features=10,
            convergence_tolerance=1e-7,
            use_lambdas="best",
            n_cpus=n_cpus
        )

        all_y_true_alpha, all_y_pred_prob_alpha = [], []
        aucs_alpha = []

        for train_idx, test_idx in folds:
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

            model_alpha.fit(X_train, y_train)
            y_pred_prob_alpha = model_alpha.predict(X_test)

            all_y_true_alpha.extend(y_test)
            all_y_pred_prob_alpha.extend(y_pred_prob_alpha)

            # AUC per fold (guard: requires both classes)
            if len(np.unique(y_test)) > 1:
                fpr, tpr, _ = roc_curve(y_test, y_pred_prob_alpha)
                aucs_alpha.append(auc(fpr, tpr))

        all_y_true_alpha = np.array(all_y_true_alpha)
        all_y_pred_prob_alpha = np.array(all_y_pred_prob_alpha)

        # F1 on concatenated predictions (threshold-independent sweep)
        precisions_alpha, recalls_alpha, thresholds_alpha = precision_recall_curve(all_y_true_alpha, all_y_pred_prob_alpha)
        f1_scores_alpha = 2 * (precisions_alpha * recalls_alpha) / (precisions_alpha + recalls_alpha + 1e-9)
        best_f1_alpha = float(np.max(f1_scores_alpha))
        mean_auc_alpha = float(np.mean(aucs_alpha)) if len(aucs_alpha) else float("nan")

        reg_results.append({
            "beta_multiplier": reg,
            "best_f1": best_f1_alpha,
            "mean_auc": mean_auc_alpha,
        })

    # Save tuning table
    reg_df = pd.DataFrame(reg_results)
    reg_df.to_csv(os.path.join(outputdir, f"Regularization_{sp}.csv"), index=False)

    # Plot F1 vs beta_multiplier
    plt.figure(figsize=(10, 5))
    plt.plot(reg_df["beta_multiplier"], reg_df["best_f1"], marker="o", label="Best F1")
    plt.xscale("log")
    plt.xlabel("beta_multiplier (log scale)")
    plt.ylabel("Best F1")
    plt.title(f"Effect of Regularization on F1 ({cv_name})")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outputdir, f"Regularization_F1_{sp}.png"))
    plt.show()

    # Plot AUC vs beta_multiplier (diagnostic; may be NaN if folds were single-class)
    plt.figure(figsize=(10, 5))
    plt.plot(reg_df["beta_multiplier"], reg_df["mean_auc"], marker="o", color="orange", label="Mean AUC")
    plt.xscale("log")
    plt.xlabel("beta_multiplier (log scale)")
    plt.ylabel("Mean AUC")
    plt.title(f"Effect of Regularization on AUC ({cv_name})")
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(outputdir, f"Regularization_AUC_{sp}.png"))
    plt.show()

    # ---------- select best regularization ----------
    sel = "best_f1" if selection_metric not in {"best_f1", "mean_auc"} else selection_metric
    if sel == "mean_auc":
        # if all AUCs are NaN (e.g., tiny species), fall back to F1
        if np.all(np.isnan(reg_df["mean_auc"].values)):
            sel = "best_f1"

    best_entry = max(reg_results, key=lambda r: r[sel] if not np.isnan(r[sel]) else -np.inf)
    best_beta = best_entry["beta_multiplier"]
    best_score = best_entry[sel]
    print(f"✅ Selected beta_multiplier={best_beta} (CV {sel}={best_score:.4f})")

    # Persist selection
    with open(os.path.join(outputdir, f"best_beta_{sp}.txt"), "w") as f:
        f.write(str(best_beta))

    # ---------- final refit on ALL data ----------
    final_model = elapid.MaxentModel(
        feature_types=feature_types,
        tau=base_tau,
        clamp=True,
        scorer="roc_auc",
        beta_multiplier=best_beta,
        beta_lqp=1.0,
        beta_hinge=1.0,
        beta_threshold=1.0,
        beta_categorical=1.0,
        n_hinge_features=10,
        n_threshold_features=10,
        convergence_tolerance=1e-7,
        use_lambdas="best",
        n_cpus=n_cpus
    )

    final_model.fit(X, y)
    y_pred_prob_full = final_model.predict(X)

    # ---------- final diagnostics ----------
    precisions_f, recalls_f, thresholds_f = precision_recall_curve(y, y_pred_prob_full)
    f1_scores_f = 2 * (precisions_f * recalls_f) / (precisions_f + recalls_f + 1e-9)
    idx_f = int(np.argmax(f1_scores_f))
    final_threshold = thresholds_f[idx_f]
    final_f1 = float(f1_scores_f[idx_f])

    if len(np.unique(y)) > 1:
        fpr_f, tpr_f, roc_thresholds_f = roc_curve(y, y_pred_prob_full)
        final_auc = float(auc(fpr_f, tpr_f))
    else:
        final_auc = float("nan")  # AUC undefined with single class in full data

    final_precision = precision_score(y, (y_pred_prob_full >= final_threshold).astype(int))
    final_recall = recall_score(y, (y_pred_prob_full >= final_threshold).astype(int))
    final_logloss = log_loss(y, y_pred_prob_full, labels=[0, 1])
    prevalence = y.mean()

    print(f"🏁 Final tuned model (beta_multiplier={best_beta})")
    print(f"AUC={final_auc:.4f}  Precision={final_precision:.4f}  Recall={final_recall:.4f}  F1={final_f1:.4f}")
    print(f"Best threshold={final_threshold:.4f}  Log-loss={final_logloss:.4f}  Prevalence={prevalence:.4f}")

    #Save raw predictor names used at training for plug-and-play inference
    pd.Series(X.columns).to_csv(os.path.join(outputdir, f"predictors_{sp}.txt"),
                                index=False, header=False)

    # Save final model & metrics BEFORE importance (so refits for importance don’t overwrite it)
    joblib.dump(final_model, os.path.join(outputdir, f"elapid_maxent_model_tuned_{sp}.pkl"))
    np.savetxt(os.path.join(outputdir, f"accuracy_tuned_{sp}.csv"),
               [['beta_multiplier', best_beta],
                ['SelectionMetric', sel],
                ['AUC', final_auc],
                ['Precision', final_precision],
                ['Recall', final_recall],
                ['F1', final_f1],
                ['BestThreshold', final_threshold],
                ['LogLoss', final_logloss],
                ['Prevalence', prevalence]],
               delimiter=',', fmt='%s')

    # Final PR/ROC plots
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(recalls_f, precisions_f, label="PR (final tuned)", color="navy")
    plt.scatter(recalls_f[idx_f], precisions_f[idx_f], color="red", s=80, label=f"Best F1={final_f1:.2f}")
    plt.title(f"PR Curve (beta={best_beta})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()

    plt.subplot(1, 2, 2)
    if not np.isnan(final_auc):
        # Mark point closest to final_best_threshold (optional)
        roc_thresholds_f = np.linspace(0, 1, len(recalls_f))  # dummy, just for plotting label anchor
        plt.plot(*roc_curve(y, y_pred_prob_full)[:2], label=f"ROC (AUC={final_auc:.2f})", color="darkorange")
    else:
        plt.text(0.5, 0.5, "AUC undefined (single class in y)", ha="center", va="center")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title(f"ROC Curve (beta={best_beta})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(outputdir, f"Curves_tuned_{sp}.png"))
    plt.show()

    # ---------- robust feature importance ----------
    # (1) Try native MaxEnt lambdas + feature names
    lambdas = getattr(final_model, "lambdas_", None)
    names = getattr(final_model, "feature_names_", None)

    if lambdas is not None:
        lambdas = np.asarray(lambdas).ravel()
        if names is None or len(names) != len(lambdas):
            names = [f"feature_{i}" for i in range(len(lambdas))]
        df_imp = pd.DataFrame({
            "feature": names,
            "lambda_weight": lambdas,
            "abs_importance": np.abs(lambdas)
        }).sort_values("abs_importance", ascending=True)
        df_imp.to_csv(os.path.join(outputdir, f"FeatureImportance_Lambdas_{sp}.csv"), index=False)
        plt.figure(figsize=(10, max(6, 0.3 * len(names))))
        plt.barh(df_imp["feature"], df_imp["lambda_weight"], color="skyblue")
        plt.axvline(0, color="gray", linewidth=1)
        plt.title("MaxEnt Feature Importance (lambda weights)")
        plt.xlabel("Lambda weight (positive ↑ suitability)")
        plt.tight_layout()
        plt.savefig(os.path.join(outputdir, f"FeatureImportance_Lambdas_{sp}.png"))
        plt.show()

    else:
        # (2) Permutation importance fallback, safe for single-class holdout
        # Use a fresh model instance so we don't alter the saved final_model
        model_pi = elapid.MaxentModel(
            feature_types=feature_types,
            tau=base_tau,
            clamp=True,
            scorer="roc_auc",
            beta_multiplier=best_beta,
            beta_lqp=1.0,
            beta_hinge=1.0,
            beta_threshold=1.0,
            beta_categorical=1.0,
            n_hinge_features=10,
            n_threshold_features=10,
            convergence_tolerance=1e-7,
            use_lambdas="best",
            n_cpus=n_cpus
        )

        # Stratified split to maximize chance both classes appear
        try:
            X_train_pi, X_holdout_pi, y_train_pi, y_holdout_pi = train_test_split(
                X, y, test_size=0.2, random_state=random_state, stratify=y
            )
        except ValueError:
            # If stratify fails (e.g., no positives), fall back to simple split
            split = int(0.8 * len(X))
            X_train_pi, X_holdout_pi = X.iloc[:split].copy(), X.iloc[split:].copy()
            y_train_pi, y_holdout_pi = y.iloc[:split].copy(), y.iloc[split:].copy()

        model_pi.fit(X_train_pi, y_train_pi)

        # Robust baseline: prefer AUC (if both classes), else use −log_loss
        def baseline_score(y_true, y_prob):
            if len(np.unique(y_true)) > 1:
                return roc_auc_score(y_true, y_prob)
            return -log_loss(y_true, y_prob, labels=[0, 1])

        base_score = baseline_score(y_holdout_pi, model_pi.predict(X_holdout_pi))

        # Robust scorer for permutation importance
        def robust_prob_scorer(est, X_eval, y_eval):
            y_prob = est.predict(X_eval)  # elapid returns 1D prob of presence
            if len(np.unique(y_eval)) > 1:
                return roc_auc_score(y_eval, y_prob)
            return -log_loss(y_eval, y_prob, labels=[0, 1])

        perm = permutation_importance(
            model_pi,
            X_holdout_pi,
            y_holdout_pi,
            scoring=robust_prob_scorer,
            n_repeats=10
        )

        imp = pd.DataFrame({
            "feature": X.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std
        }).sort_values("importance_mean", ascending=True)

        imp.to_csv(os.path.join(outputdir, f"PermutationImportance_{sp}.csv"), index=False)
        plt.figure(figsize=(10, max(6, 0.3 * len(imp))))
        plt.barh(imp["feature"], imp["importance_mean"], xerr=imp["importance_std"], color="steelblue")
        plt.axvline(0, color="gray", linewidth=1)
        metric_label = "AUC" if len(np.unique(y_holdout_pi)) > 1 else "−LogLoss"
        plt.title(f"Permutation Importance ({metric_label} change). Baseline={base_score:.3f}")
        plt.xlabel(f"Mean importance (Δ{metric_label})")
        plt.tight_layout()
        plt.savefig(os.path.join(outputdir, f"PermutationImportance_{sp}.png"))
        plt.show()

    print("✅ Tuned final MaxEnt (PPP-equivalent) model saved, with diagnostics & importance.")
    return {
        "best_beta": best_beta,
        "selection_metric": sel,
        "final_auc": final_auc,
        "final_f1": final_f1,
        "final_precision": final_precision,
        "final_recall": final_recall,
        "final_threshold": final_threshold,
        "final_logloss": final_logloss,
        "prevalence": prevalence,
        "cv_name": cv_name,
        "reg_results": reg_results
    }

In [ ]:
# Prepare X, y, coords as you already do, then:
results = fit_maxent_with_tuning(
    X=X,
    y=y,
    coords=coords,
    outputdir=outputdir,
    sp=sp,
    selection_metric="best_f1"  # or "mean_auc"
)

print(results)
